# 27 — Retrieval-Only Smoke (No Gemini)

Use this while the Gemini free-tier quota is exhausted.

Checks:
1. Basic top-k retrieval
2. Source / facet filters
3. Exclude already-seen IDs
4. Facet coverage for eval questions (retrieval stage only)

No LLM calls.

In [ ]:
import os
import sys
import json

project_root = os.path.abspath("..")

if project_root not in sys.path:
    sys.path.insert(0, project_root)

In [ ]:
from rag_icot.components.retriever import Retriever
from rag_icot.evaluation import load_eval_dataset
from rag_icot.evaluation.metrics import (
    compute_facet_coverage,
    facet_recall,
    source_diversity,
)

In [ ]:
retriever = Retriever()
print("Retriever ready (embedding model only, no Gemini)")

## 1) Basic Mirai retrieval

In [ ]:
query = "Mirai botnet IoT infection behaviour"
results = retriever.retrieve(query, k=5)

for i, (doc_id, text, meta) in enumerate(
    zip(
        results["ids"][0],
        results["documents"][0],
        results["metadatas"][0],
    ),
    start=1,
):
    preview = (text or "")[:140].replace("\n", " ")
    print(f"[{i}] {doc_id} | {meta.get('source')} | {preview}...")

## 2) Source / facet filters

In [ ]:
checks = [
    ("IoT23 source", dict(source="IoT23")),
    ("MITRE source", dict(source="MITRE")),
    ("behaviour facet", dict(facet="behaviour")),
    ("vulnerability facet", dict(facet="vulnerability")),
    ("exploit facet", dict(facet="exploit")),
    ("technique facet", dict(facet="technique")),
]

for name, kwargs in checks:
    res = retriever.retrieve("IoT security", k=3, **kwargs)
    sources = [m.get("source") for m in res["metadatas"][0]]
    types = [m.get("document_type") for m in res["metadatas"][0]]
    print(f"{name:22} ids={res['ids'][0]}")
    print(f"{'':22} sources={sources} types={types}")

## 3) Exclude already-seen IDs

In [ ]:
first = retriever.retrieve("Mirai", k=3, source="IoT23")
seen = set(first["ids"][0])
second = retriever.retrieve(
    "Mirai",
    k=3,
    source="IoT23",
    exclude_ids=seen,
)

print("First:", first["ids"][0])
print("Second (excluded first):", second["ids"][0])
overlap = seen.intersection(second["ids"][0])
print("Overlap:", overlap)
assert not overlap, "Exclude-IDs failed"
print("Exclude-IDs OK")

## 4) Retrieval facet recall on eval subset

Measures whether a single retrieval pass covers the question's required facets.
This is **not** full ICOT quality — only retrieval coverage.

In [ ]:
dataset_path = os.path.join(
    project_root,
    "datasets",
    "evaluation",
    "iot_security_eval_v1.json",
)

questions = load_eval_dataset(dataset_path)

# Broader than LLM smoke — still free (no Gemini)
subset_ids = [
    "q001", "q002", "q011", "q019", "q021",
    "q025", "q031", "q033", "q041", "q050",
]
subset = [q for q in questions if q.id in subset_ids]

rows = []

for q in subset:
    res = retriever.retrieve(q.question, k=5)
    docs = [
        {
            "id": doc_id,
            "text": text,
            "metadata": meta or {},
        }
        for doc_id, text, meta in zip(
            res["ids"][0],
            res["documents"][0],
            res["metadatas"][0],
        )
    ]

    covered = compute_facet_coverage(docs)
    recall = facet_recall(q.required_facets, covered)

    row = {
        "id": q.id,
        "category": q.category,
        "required_facets": q.required_facets,
        "covered_facets": covered,
        "facet_recall": recall,
        "sources": source_diversity(docs),
        "top_ids": [d["id"] for d in docs],
    }
    rows.append(row)

    print(
        f"{q.id} recall={recall:.2f} "
        f"required={q.required_facets} covered={covered} "
        f"sources={row['sources']}"
    )

In [ ]:
avg_recall = sum(r["facet_recall"] for r in rows) / max(len(rows), 1)
print(f"Average retrieval facet recall ({len(rows)} qs): {avg_recall:.3f}")

weak = [r for r in rows if r["facet_recall"] < 1.0]
print(f"Questions with incomplete facet coverage: {len(weak)}")
for r in weak:
    missing = sorted(set(r["required_facets"]) - set(r["covered_facets"]))
    print(f"  {r['id']}: missing={missing}")

In [ ]:
out_dir = os.path.join(project_root, "artifacts", "evaluation")
os.makedirs(out_dir, exist_ok=True)

out_path = os.path.join(out_dir, "retrieval_only_smoke.json")

with open(out_path, "w", encoding="utf-8") as f:
    json.dump(
        {
            "average_facet_recall": avg_recall,
            "rows": rows,
        },
        f,
        indent=2,
        ensure_ascii=False,
    )

print("Saved", out_path)

## Pass criteria

- Mirai query returns IoT-23 Mirai doc near top
- Source/facet filters return matching metadata
- Exclude-IDs has zero overlap
- Multi-facet questions often show incomplete coverage in **single-pass** retrieval
  (this motivates iterative facet ICOT when Gemini quota returns)